In [ ]:
# Import required libraries for data manipulation, preprocessing, and model evaluation.

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, RFECV, SelectFromModel
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import cross_val_score
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Configure Scikit-Learn to output Pandas DataFrames
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
# Upload housing_iteration_6_regression.csv to your own Google Drive
# Paste your file's ID below
file_id = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"
url = f"https://drive.google.com/uc?export=download&id={file_id}"
df = pd.read_csv(url)
df = df.set_index("Id")

In [ ]:
# Separate target variable ('SalePrice') from features
X = df
y = X.pop("SalePrice")


# Converts MSSubClass to string type for One-Hot Encoding, as these numeric codes are not hierarchical.
X["MSSubClass"] = X["MSSubClass"].astype(str)

# Apply natural logarithm transformation to target variable y to optimize Kaggle's RMSE on log(SalePrice) evaluation metric and stabilize target variance
y = np.log(y)

# Splits the dataset into training (80%) and testing (20%) sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

In [ ]:
# -----------------------------
# Preprocessing Pipelines
# -----------------------------
# Identify numerical, nominal and ordinal columns
num_cols = [
    "YrSold", "MiscVal", "PoolArea", "ScreenPorch", "3SsnPorch",
    "EnclosedPorch", "OpenPorchSF", "WoodDeckSF", "GarageArea",
    "GarageCars", "Fireplaces", "TotRmsAbvGrd", "KitchenAbvGr",
    "BedroomAbvGr", "HalfBath", "FullBath", "BsmtHalfBath",
    "BsmtFullBath", "GrLivArea", "1stFlrSF", "2ndFlrSF",
    "TotalBsmtSF", "YearRemodAdd", "YearBuilt", "OverallCond",
    "OverallQual", "LotArea", "LotFrontage",
    "MoSold", "GarageYrBlt", "LowQualFinSF", "BsmtFinSF2",
    "BsmtUnfSF", "BsmtFinSF1", "MasVnrArea",
]

nominal_cols = [
    "SaleCondition", "SaleType", "MiscFeature", "Fence", "GarageType",
    "CentralAir", "Heating", "Foundation", "MasVnrType", "Exterior1st",
    "RoofMatl", "RoofStyle", "HouseStyle", "Neighborhood", "LotConfig",
    "LandContour", "Street", "MSZoning",
    "Condition1",
    "MSSubClass",
    "Electrical", "Exterior2nd", "BldgType", "Condition2", "Alley", "Utilities",
    "LandSlope", "LotShape",
]

ordinal_cols = [
    "PoolQC", "PavedDrive", "GarageCond", "GarageQual", "GarageFinish",
    "FireplaceQu", "Functional", "KitchenQual", "HeatingQC",
    "BsmtFinType1", "BsmtExposure", "BsmtCond", "BsmtQual",
    "ExterCond", "ExterQual",
    "BsmtFinType2",
]

ordinal_categories = [
    ["N_A", "Fa", "TA", "Gd", "Ex"],                       # PoolQC
    ["N", "P", "Y"],                                       # PavedDrive
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # GarageCond
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # GarageQual
    ["N_A", "Unf", "RFn", "Fin"],                           # GarageFinish
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # FireplaceQu
    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],  # Functional
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # KitchenQual
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # HeatingQC
    ["N_A", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],      # BsmtFinType1
    ["N_A", "No", "Mn", "Av", "Gd"],                        # BsmtExposure
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # BsmtCond
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # BsmtQual
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # ExterCond
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # ExterQual
    ["N_A", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],      # BsmtFinType2
]

# Construct pipeline for numerical features using mean imputation to handle missing values
# SimpleImputer(strategy="mean"): Replaces missing numerical values (NaN) with the average of that feature column
num_pipe = make_pipeline(
    SimpleImputer(strategy="mean")
)


# Construct pipeline for nominal categorical features (unordered categories, e.g., Neighborhood)
# OneHotEncoder: Converts nominal categories into binary dummy variables (0s and 1s); ignores unseen categories during testing

nominal_pipe = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="N_A"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False)
)


# Construct pipeline for ordinal categorical features
# OrdinalEncoder: Maps ordered categories to integer values based on predefined rankings (ordinal_categories); maps unknown values to -1

ordinal_pipe = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="N_A"),
    OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1)
)

# Combine numerical, ordinal, and nominal feature pipelines into a unified ColumnTransformer
# Applies each transformation pipeline exclusively to its designated feature column list

preprocessor = make_column_transformer(
    (num_pipe, num_cols),
    (ordinal_pipe, ordinal_cols),
    (nominal_pipe, nominal_cols)
)



In [ ]:
# Define helper function to fit models and evaluate performance using R-squared metric on the test set

def score_models(feat_select_method, tree_pipe, knn_pipe):
    # Fit Decision Tree and KNN pipelines on the training set
    tree_pipe.fit(X_train, y_train)
    knn_pipe.fit(X_train, y_train)
    # Calculate and store R-squared scores for both models along with the feature selection method name
    scores = {
        'Feature Selection': feat_select_method,
        'Decision Tree': r2_score(y_test, tree_pipe.predict(X_test)),
        'KNN': r2_score(y_test, knn_pipe.predict(X_test))
    }
    return scores

In [ ]:
# Initialize list to collect evaluation scores across different feature selection strategies
method_scores = []


# Construct baseline pipeline for Decision Tree Regressor using preprocessed raw features
base_tree = make_pipeline(
    preprocessor,
    DecisionTreeRegressor(random_state=42)
)

# Construct baseline pipeline for KNN Regressor, including MinMaxScaler as distance-based algorithms require feature scaling
base_knn = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    KNeighborsRegressor(n_neighbors=6)
)
# Evaluate baseline models, append scores to performance log, and display results in a structured DataFrame

method_scores.append(score_models('Baseline', base_tree, base_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.77894,0.716427


In [ ]:
#Feature selection based only on features

# Construct Decision Tree pipeline applying VarianceThreshold to remove low-variance features (threshold=0.02) after scaling
var0_02_tree = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0.02),
    DecisionTreeRegressor(random_state=42)
)

# Construct KNN pipeline applying VarianceThreshold filtering (threshold=0.02) to scale features before distance calculation
var0_02_knn = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0.02),
    KNeighborsRegressor(n_neighbors=6)
)

# Evaluate models with 0.02 variance threshold filtering, append scores, and update performance summary DataFrame
method_scores.append(score_models('Variance Threshold (0.02)', var0_02_tree, var0_02_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814


In [ ]:
# Variance Threshold 0.001

var0_001_tree = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0.001),
    DecisionTreeRegressor(random_state=42)
)
var0_001_knn = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0.001),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Variance Threshold (0.001)', var0_001_tree, var0_001_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427


In [ ]:
# Variance Threshold 0
var0_tree = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0),
    DecisionTreeRegressor(random_state=42)
)
var0_knn = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    VarianceThreshold(threshold=0),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Variance Threshold (0)', var0_tree, var0_knn))
pd.DataFrame(method_scores)

# variance threshold doesnt help

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427


In [ ]:
# Perform collinearity analysis by building correlation matrix on preprocessed training data
X_train_pre = preprocessor.fit_transform(X_train, y_train)
corrMatrix = X_train_pre.corr().abs()

# Define threshold (0.95) to identify pairs of highly collinear features
correlation_threshold = 0.95
highly_correlated_columns = []
num_features = len(corrMatrix.columns)
for i in range(num_features):
    for j in range(i + 1, num_features):
        if corrMatrix.iloc[i, j] >= correlation_threshold:
            highly_correlated_columns.append(
                (corrMatrix.columns[i], corrMatrix.columns[j],
                 f"correlation = {round(corrMatrix.iloc[i, j], 2)}")
            )


# Print identified highly correlated feature pairs and extracted drop list
print("Highly correlated pairs:", highly_correlated_columns)

to_drop = [a for a, b, c in highly_correlated_columns]
print("\nColumns to drop:", to_drop)

Highly correlated pairs: [('pipeline-2__GarageCond', 'pipeline-2__GarageQual', 'correlation = 0.96'), ('pipeline-3__SaleCondition_Partial', 'pipeline-3__SaleType_New', 'correlation = 0.98'), ('pipeline-3__MiscFeature_N_A', 'pipeline-3__MiscFeature_Shed', 'correlation = 0.95'), ('pipeline-3__CentralAir_N', 'pipeline-3__CentralAir_Y', 'correlation = 1.0'), ('pipeline-3__Exterior1st_CBlock', 'pipeline-3__Exterior2nd_CBlock', 'correlation = 1.0'), ('pipeline-3__Exterior1st_CemntBd', 'pipeline-3__Exterior2nd_CmentBd', 'correlation = 0.98'), ('pipeline-3__Exterior1st_MetalSd', 'pipeline-3__Exterior2nd_MetalSd', 'correlation = 0.97'), ('pipeline-3__Exterior1st_VinylSd', 'pipeline-3__Exterior2nd_VinylSd', 'correlation = 0.98'), ('pipeline-3__Street_Grvl', 'pipeline-3__Street_Pave', 'correlation = 1.0'), ('pipeline-3__MSSubClass_190', 'pipeline-3__BldgType_2fmCon', 'correlation = 0.98'), ('pipeline-3__MSSubClass_90', 'pipeline-3__BldgType_Duplex', 'correlation = 1.0'), ('pipeline-3__Utilities_A

In [ ]:
# Construct Decision Tree pipeline that drops highly collinear features (correlation >= 0.95) using ColumnTransformer
corr_tree = make_pipeline(
    preprocessor,
    ColumnTransformer([("corrdropper", "drop", to_drop)], remainder="passthrough"),
    DecisionTreeRegressor(random_state=42)
)

# Construct KNN pipeline that drops collinear features before scaling remaining features for distance calculations
corr_knn = make_pipeline(
    preprocessor,
    ColumnTransformer([("corrdropper", "drop", to_drop)], remainder="passthrough"),
    MinMaxScaler(),
    KNeighborsRegressor(n_neighbors=6)
)
# Evaluate models after removing highly collinear features, append scores, and display updated performance summary DataFrame
method_scores.append(score_models('Collinearity Threshold (0.95)', corr_tree, corr_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884


In [ ]:
# Perform collinearity analysis with a stricter correlation threshold (0.90) on preprocessed training data
X_train_pre = preprocessor.fit_transform(X_train, y_train)
corrMatrix = X_train_pre.corr().abs()


# Identify feature pairs exceeding the 0.90 absolute correlation threshold
correlation_threshold = 0.90
highly_correlated_columns = []
num_features = len(corrMatrix.columns)
for i in range(num_features):
    for j in range(i + 1, num_features):
        if corrMatrix.iloc[i, j] >= correlation_threshold:
            highly_correlated_columns.append(
                (corrMatrix.columns[i], corrMatrix.columns[j],
                 f"correlation = {round(corrMatrix.iloc[i, j], 2)}")
            )

# Extract unique list of features targeted for removal and display total count
to_drop_90 = [a for a, b, c in highly_correlated_columns]
print("Number of columns to drop:", len(set(to_drop_90)))


# Construct Decision Tree pipeline that drops features exceeding 0.90 correlation
corr_tree_90 = make_pipeline(
    preprocessor,
    ColumnTransformer([("corrdropper", "drop", to_drop_90)], remainder="passthrough"),
    DecisionTreeRegressor(random_state=42)
)
# Construct KNN pipeline that drops collinear features before scaling remaining features for distance calculations
corr_knn_90 = make_pipeline(
    preprocessor,
    ColumnTransformer([("corrdropper", "drop", to_drop_90)], remainder="passthrough"),
    MinMaxScaler(),
    KNeighborsRegressor(n_neighbors=6)
)

# Evaluate models after applying 0.90 collinearity threshold, append scores, and display updated performance summary DataFrame

method_scores.append(score_models('Collinearity Threshold (0.90)', corr_tree_90, corr_knn_90))
pd.DataFrame(method_scores)

Number of columns to drop: 19


,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365


In [ ]:
# correlation threshold 0.85
correlation_threshold = 0.85
highly_correlated_columns_85 = []
num_features = len(corrMatrix.columns)
for i in range(num_features):
    for j in range(i + 1, num_features):
        if corrMatrix.iloc[i, j] >= correlation_threshold:
            highly_correlated_columns_85.append(
                (corrMatrix.columns[i], corrMatrix.columns[j],
                 f"correlation = {round(corrMatrix.iloc[i, j], 2)}")
            )

to_drop_85 = [a for a, b, c in highly_correlated_columns_85]
print("Number of columns to drop:", len(set(to_drop_85)))

corr_tree_85 = make_pipeline(
    preprocessor,
    ColumnTransformer([("corrdropper", "drop", to_drop_85)], remainder="passthrough"),
    DecisionTreeRegressor(random_state=42)
)
corr_knn_85 = make_pipeline(
    preprocessor,
    ColumnTransformer([("corrdropper", "drop", to_drop_85)], remainder="passthrough"),
    MinMaxScaler(),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Collinearity Threshold (0.85)', corr_tree_85, corr_knn_85))
pd.DataFrame(method_scores)



Number of columns to drop: 26


,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522


In [ ]:
# Construct Decision Tree pipeline selecting top 30 features (k=30) based on f_regression univariate statistical test after scaling

kbest_tree = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=30),
    DecisionTreeRegressor(random_state=42)
)

# Construct KNN pipeline selecting top 30 features (k=30) using f_regression scoring prior to distance calculation
kbest_knn = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=30),
    KNeighborsRegressor(n_neighbors=6)
)
# Evaluate models using SelectKBest (k=30), append scores to performance log, and display updated results DataFrame
method_scores.append(score_models('Select K Best (k=30)', kbest_tree, kbest_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935


In [ ]:
# best k value , k=40

kbest_tree_40 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=40),
    DecisionTreeRegressor(random_state=42)
)
kbest_knn_40 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=40),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Select K Best (k=40)', kbest_tree_40, kbest_knn_40))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935
8,Select K Best (k=40),0.762368,0.735043


In [ ]:
# best k value , k=20

kbest_tree_20 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=20),
    DecisionTreeRegressor(random_state=42)
)
kbest_knn_20 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=20),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Select K Best (k=20)', kbest_tree_20, kbest_knn_20))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935
8,Select K Best (k=40),0.762368,0.735043
9,Select K Best (k=20),0.729742,0.807928


In [ ]:
# best k value , k=10

kbest_tree_10 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=10),
    DecisionTreeRegressor(random_state=42)
)
kbest_knn_10 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=10),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Select K Best (k=10)', kbest_tree_10, kbest_knn_10))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935
8,Select K Best (k=40),0.762368,0.735043
9,Select K Best (k=20),0.729742,0.807928


In [ ]:
# best k value , k=5

kbest_tree_5 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=5),
    DecisionTreeRegressor(random_state=42)
)
kbest_knn_5 = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SelectKBest(score_func=f_regression, k=5),
    KNeighborsRegressor(n_neighbors=6)
)

method_scores.append(score_models('Select K Best (k=5)', kbest_tree_5, kbest_knn_5))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935
8,Select K Best (k=40),0.762368,0.735043
9,Select K Best (k=20),0.729742,0.807928


In [ ]:
# Construct Decision Tree pipeline using SelectFromModel to select features based on feature importances from an embedded Decision Tree
SFM_tree = make_pipeline(
    preprocessor,
    SelectFromModel(DecisionTreeRegressor(random_state=42)),
    DecisionTreeRegressor(random_state=42)
)

# Construct KNN pipeline using tree-based feature selection before applying MinMaxScaler and distance calculations
SFM_knn = make_pipeline(
    preprocessor,
    SelectFromModel(DecisionTreeRegressor(random_state=42)),
    MinMaxScaler(),
    KNeighborsRegressor(n_neighbors=6)
)
# Evaluate models with SelectFromModel feature selection, append scores to log, and display summary DataFrame
method_scores.append(score_models('Select From Model-DecisionTree_R', SFM_tree, SFM_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935
8,Select K Best (k=40),0.762368,0.735043
9,Select K Best (k=20),0.729742,0.807928


In [ ]:
# Construct Decision Tree pipeline using Recursive Feature Elimination with Cross-Validation (RFECV) to select the optimal subset of features based on Decision Tree feature importances
RFE_tree = make_pipeline(
    preprocessor,
    RFECV(DecisionTreeRegressor(random_state=42)),
    DecisionTreeRegressor(random_state=42)
)

# Construct KNN pipeline using RFECV feature selection prior to scaling and distance-based neighborhood calculations
RFE_knn = make_pipeline(
    preprocessor,
    RFECV(DecisionTreeRegressor(random_state=42)),
    MinMaxScaler(),
    KNeighborsRegressor(n_neighbors=6)
)
# Evaluate models with RFECV feature selection, append scores to log, and display updated performance summary DataFrame
method_scores.append(score_models('RFECV- DecisionTree_R', RFE_tree, RFE_knn))
pd.DataFrame(method_scores)

,Feature Selection,Decision Tree,KNN
0,Baseline,0.778940,0.716427
1,Variance Threshold (0.02),0.702183,0.721814
2,Variance Threshold (0.001),0.781979,0.716427
3,Variance Threshold (0),0.778969,0.716427
4,Collinearity Threshold (0.95),0.786059,0.694884
5,Collinearity Threshold (0.90),0.801299,0.711365
6,Collinearity Threshold (0.85),0.800696,0.707522
7,Select K Best (k=30),0.772537,0.747935
8,Select K Best (k=40),0.762368,0.735043
9,Select K Best (k=20),0.729742,0.807928


In [ ]:
# Extract features selected by RFECV step from the fitted KNN pipeline and identify eliminated feature columns
rfecv_selected_features = RFE_knn['rfecv'].get_feature_names_out()
all_features = X_train_pre.columns
rfecv_drop_cols = [c for c in all_features if c not in rfecv_selected_features]

# Display total number of features flagged for removal by RFECV
print("Number of columns to drop with RFECV:", len(rfecv_drop_cols))


# Prepare feature subset for tree-based models (e.g., XGBoost) using 0.90 collinearity threshold drop list
X_train_tree = X_train_pre.drop(columns=to_drop_90)

# Prepare feature subset for distance and linear models (e.g., KNN, Linear Regression) using RFECV drop list
X_train_linear = X_train_pre.drop(columns=rfecv_drop_cols)

# Display final column counts for model family specific feature subsets
print("Remaining columns for XGBoost:", X_train_tree.shape[1])
print("Remaining columns for KNN/Linear Regression:", X_train_linear.shape[1])



Number of columns to drop with RFECV: 240
Remaining columns for XGBoost: 229
Remaining columns for KNN/Linear Regression: 8


In [ ]:
#XGBoost Model
# Construct pipeline for XGBoost Regressor using tree-specific feature subset
xgb_pipe = make_pipeline(
    xgb.XGBRegressor(random_state=42)
)

# Define hyperparameter grid to tune tree depth, learning rate, and number of estimators
xgb_params = {
    "xgbregressor__n_estimators": [100, 300, 500],
    "xgbregressor__max_depth": [2, 3, 4, 6],
    "xgbregressor__learning_rate": [0.01, 0.05, 0.1]
}

# Configure 5-fold cross-validation grid search optimizing for negative RMSE on log-transformed target
xgb_grid = GridSearchCV(
    xgb_pipe,
    xgb_params,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

# Fit grid search on tree-based feature subset
xgb_grid.fit(X_train_tree, y_train)

# Display optimal hyperparameter combination and best cross-validated RMSE score
print("XGBoost - best parameters:", xgb_grid.best_params_)
print("XGBoost - best CV RMSE (log):", -xgb_grid.best_score_)

XGBoost - best parameters: {'xgbregressor__learning_rate': 0.1, 'xgbregressor__max_depth': 3, 'xgbregressor__n_estimators': 500}
XGBoost - best CV RMSE (log): 0.1331972828694424


In [ ]:
# Linear Regression Model

lr_pipe = make_pipeline(MinMaxScaler(), LinearRegression())

lr_scores = cross_val_score(
    lr_pipe,
    X_train_linear,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("Linear Regression - CV RMSE (log):", -lr_scores.mean())

Linear Regression - CV RMSE (log): 0.17859610835041012
